In [2]:
# Installing necessary libraries
!pip -q install -U transformers trl

In [3]:
# For generating the jsonl file in the desired format
import json
from pathlib import Path

data_dir = Path("/kaggle/input/datasets/deepthiajith/ricedisease-traindataset/Rice Disease Dataset")
jsonl_file = "rice_disease_dataset.jsonl"

classes = [d.name for d in data_dir.iterdir() if d.is_dir()]
class_list_str = ", ".join(classes)

instruction = f"Classify this image into one of the following categories: {class_list_str}. Output only the category name."

with open(jsonl_file, "w") as f:
    for class_name in classes:
        class_dir = data_dir / class_name
        for img_path in class_dir.glob("*.*"):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                record = {
                    "image_path": str(img_path),
                    "label": class_name,
                    "instruction": instruction
                }
                f.write(json.dumps(record) + "\n")

In [4]:
# Necessary import and setting up environment variables
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import cv2
import gc
from PIL import Image
from datasets import load_dataset
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

In [ ]:
torch.cuda.empty_cache()
gc.collect()

raw_dataset = load_dataset("json", data_files="rice_disease_dataset.jsonl", split="train")

model_id = "Qwen/Qwen3-VL-4B-Instruct"

# Setting up the Processor
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=28 * 28,
    max_pixels=224 * 224,
)

processor.tokenizer.pad_token    = processor.tokenizer.eos_token
processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# Importing the model
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
).to("cuda")

model.config.use_cache = False
model.enable_input_require_grads()

from transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLVisionModel
Qwen3VLVisionModel.dtype = property(lambda self: torch.float16)

# Freezing the vision layers
for name, param in model.named_parameters():
    if "visual" in name:
        param.requires_grad = False

# Ignoring Vision Layers
lora_config = LoraConfig(
    r=16, lora_alpha=16,
    target_modules=r"(?!.*visual).*\.(q_proj|v_proj|k_proj|o_proj|gate_proj|up_proj|down_proj)",
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

IMAGE_PAD_TOKEN_ID    = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
VISION_START_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<|vision_start|>")
VISION_END_TOKEN_ID   = processor.tokenizer.convert_tokens_to_ids("<|vision_end|>")
IM_START_TOKEN_ID     = processor.tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END_TOKEN_ID       = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
PAD_TOKEN_ID          = processor.tokenizer.pad_token_id

# Collator
def build_labels(input_ids: torch.Tensor) -> torch.Tensor:
    labels = input_ids.clone()

    for special_id in (PAD_TOKEN_ID, IMAGE_PAD_TOKEN_ID,
                       VISION_START_TOKEN_ID, VISION_END_TOKEN_ID):
        labels[labels == special_id] = -100

    im_start_positions = (input_ids == IM_START_TOKEN_ID).nonzero(as_tuple=True)[0]
    if len(im_start_positions) > 0:
        last_im_start = im_start_positions[-1].item()
        labels[: last_im_start + 3] = -100

    return labels


def qwen_collator(features):
    all_input_ids      = []
    all_attention_mask = []
    all_labels         = []
    all_pixel_values   = []
    all_image_grid_thw = []

    for feature in features:
        img_bgr = cv2.imread(feature["image_path"])
        img_bgr = cv2.resize(img_bgr, (224, 224))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        image   = Image.fromarray(img_rgb)

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": feature["instruction"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": feature["label"]}],
            },
        ]

        text = processor.apply_chat_template(
            conversation, add_generation_prompt=False, tokenize=False
        )

        inputs = processor(
            text=[text],
            images=[image],
            padding=False,     
            return_tensors="pt",
        )

        input_ids = inputs["input_ids"][0]          
        attn_mask = inputs["attention_mask"][0]     

        labels = build_labels(input_ids)

        all_input_ids.append(input_ids)
        all_attention_mask.append(attn_mask)
        all_labels.append(labels)

        if "pixel_values" in inputs:
            all_pixel_values.append(
                inputs["pixel_values"].squeeze(0).to(torch.float16)  
            )
        if "image_grid_thw" in inputs:
            all_image_grid_thw.append(inputs["image_grid_thw"])       

    max_len = max(ids.shape[0] for ids in all_input_ids)

    def pad_tensor(t, pad_value, length):
        return torch.nn.functional.pad(t, (0, length - t.shape[0]), value=pad_value)

    batch = {
        "input_ids":      torch.stack([pad_tensor(x, PAD_TOKEN_ID, max_len) for x in all_input_ids]),
        "attention_mask": torch.stack([pad_tensor(x, 0,            max_len) for x in all_attention_mask]),
        "labels":         torch.stack([pad_tensor(x, -100,         max_len) for x in all_labels]),
    }

    if all_pixel_values:
        batch["pixel_values"]   = torch.cat(all_pixel_values, dim=0)
        batch["image_grid_thw"] = torch.cat(all_image_grid_thw, dim=0)

    return batch

In [ ]:
# Training Args

training_args = SFTConfig(
    output_dir="qwen_t4_vlm",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=True,
    bf16=False,
    num_train_epochs=4,
    logging_steps=500,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,        
    optim="adamw_torch",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=0,
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=raw_dataset,
    args=training_args,
    processing_class=processor.tokenizer,
    data_collator=qwen_collator,
)

print("Launching training")
trainer.train(resume_from_checkpoint="/kaggle/input/models/kuchikibyakuya182/qwen-3-4b-instruct-checkpoint-16000/transformers/default/1")

In [5]:
# Inference Pipeline

from peft import PeftModel

BASE_MODEL  = "Qwen/Qwen3-VL-4B-Instruct"
ADAPTER_DIR = "/kaggle/input/models/kuchikibyakuya182/checkpoint-4th-epoch/transformers/default/1"   # LoRA checkpoint
TEST_DIR    = "/kaggle/input/datasets/kuchikibyakuya182/rice-test/test"
INSTRUCTION = f"Classify this image into one of the following categories: {class_list_str}. Output only the category name."
BATCH_SIZE  = 4
 
SUPPORTED = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
 
 
def load_model():
    processor = AutoProcessor.from_pretrained(
        BASE_MODEL, min_pixels=28 * 28, max_pixels=224 * 224
    )
    processor.tokenizer.pad_token    = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
 
    base = Qwen3VLForConditionalGeneration.from_pretrained(
        BASE_MODEL, torch_dtype=torch.float16,
        device_map="cuda", attn_implementation="sdpa",
    )
    model = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    model.eval()
    return model, processor
 
 
def collect_samples():
    samples = []
    for class_name in sorted(os.listdir(TEST_DIR)):
        class_dir = os.path.join(TEST_DIR, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if os.path.splitext(fname)[1].lower() in SUPPORTED:
                samples.append((os.path.join(class_dir, fname), class_name))
    return samples
 
 
def predict_batch(model, processor, paths):
    all_input_ids, all_masks, all_pixels, all_grids = [], [], [], []
 
    for path in paths:
        img = cv2.imread(path)
        img = cv2.cvtColor(cv2.resize(img, (224, 224)), cv2.COLOR_BGR2RGB)
 
        text = processor.apply_chat_template(
            [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": INSTRUCTION}]}],
            add_generation_prompt=True, tokenize=False,
        )
        enc = processor(text=[text], images=[Image.fromarray(img)], padding=False, return_tensors="pt")
 
        all_input_ids.append(enc["input_ids"][0])
        all_masks.append(enc["attention_mask"][0])
        if "pixel_values"   in enc: all_pixels.append(enc["pixel_values"].squeeze(0))
        if "image_grid_thw" in enc: all_grids.append(enc["image_grid_thw"])
 
    pad_id  = processor.tokenizer.pad_token_id
    max_len = max(x.shape[0] for x in all_input_ids)
 
    def pad(t, val): return torch.nn.functional.pad(t, (0, max_len - t.shape[0]), value=val)
 
    inputs = {
        "input_ids":      torch.stack([pad(x, pad_id) for x in all_input_ids]).cuda(),
        "attention_mask": torch.stack([pad(x, 0)      for x in all_masks]).cuda(),
    }
    if all_pixels:
        inputs["pixel_values"]   = torch.cat(all_pixels, dim=0).to(torch.float16).cuda()
        inputs["image_grid_thw"] = torch.cat(all_grids,  dim=0).cuda()
 
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=16, do_sample=False,
                             pad_token_id=pad_id, eos_token_id=processor.tokenizer.eos_token_id)
 
    return [
        processor.tokenizer.decode(out[i, prompt_len:], skip_special_tokens=True).strip()
        for i in range(len(paths))
    ]
 
 
def normalise(pred, classes):
    """Map free-text output to the nearest class name."""
    p = pred.lower()
    for cls in classes:
        if cls.lower() == p:     return cls   # exact
    for cls in classes:
        if cls.lower() in p:     return cls   # class name inside output
    for cls in classes:
        if p in cls.lower():     return cls   # output inside class name
    return pred                               # unmatched → counts as wrong

In [6]:
y_true = []
y_pred = []
def inference():
    print("Loading model …")
    model, processor = load_model()
 
    samples = collect_samples()
    classes = sorted({label for _, label in samples})
    print(f"Found {len(samples)} images across {len(classes)} classes: {classes}\n")
 
    correct, total = 0, 0
 
    for i in range(0, len(samples), BATCH_SIZE):
        batch   = samples[i : i + BATCH_SIZE]
        paths   = [s[0] for s in batch]
        truths  = [s[1] for s in batch]
        raw     = predict_batch(model, processor, paths)
 
        for path, truth, pred in zip(paths, truths, raw):
            matched  = normalise(pred, classes)
            hit      = matched == truth
            correct += hit
            total   += 1
            y_true.append(truth)
            y_pred.append(pred)
 
    print("─" * 60)
    print(f"  Final accuracy: {correct}/{total}  ({correct/total*100:.2f}%)")
    print("─" * 60)

In [7]:
inference()

Loading model …


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Found 465 images across 19 classes: ['Bacterial Leaf Blight', 'Bacterial Streak', 'Bakanae', 'Brown Spot', 'False Smut', 'Grassy Stunt Virus', 'Healthy Leaf', 'Hispa', 'Insect Affected', 'Leaf Blast', 'Leaf Scald', 'Leaf Smut', 'Narrow Brown Spot', 'Neck Blast', 'Ragged Stunt Virus', 'Sheath Blight', 'Sheath Rot', 'Stem Rot', 'Tungro']



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


────────────────────────────────────────────────────────────
  Final accuracy: 377/465  (81.08%)
────────────────────────────────────────────────────────────


In [8]:
# Importing classification report for F1-score
from sklearn.metrics import classification_report

In [9]:
print(classification_report(y_true, y_pred))

                       precision    recall  f1-score   support

Bacterial Leaf Blight       0.69      0.90      0.78        30
     Bacterial Streak       1.00      0.93      0.97        15
              Bakanae       0.74      0.93      0.82        15
           Brown Spot       0.69      0.83      0.76        30
           False Smut       0.88      1.00      0.94        15
   Grassy Stunt Virus       1.00      0.73      0.85        15
         Healthy Leaf       0.77      1.00      0.87        30
                Hispa       0.96      0.90      0.93        30
      Insect Affected       1.00      0.83      0.91        30
           Leaf Blast       0.44      0.83      0.57        30
           Leaf Scald       0.82      0.93      0.88        30
            Leaf Smut       1.00      0.13      0.24        30
    Narrow Brown Spot       0.81      0.57      0.67        30
           Neck Blast       1.00      1.00      1.00        30
   Ragged Stunt Virus       0.88      0.47      0.61  